# Terpedia end-to-end docking rerun

This notebook is the executable rerun path for the terpene docking reproduction. It installs the pinned chemistry environment, checks out the exact Terpedia chemistry implementation, downloads the public RCSB structures, prepares receptors, validates co-crystal redocking, runs the multi-seed docking panel, and saves a machine-readable receipt.

A target may legitimately finish as `validation_failed`, `preparation_failed`, or `cofactor_unresolved`. Those are recorded scientific outcomes, not silently omitted rows.

In [ ]:
import importlib.util, json, os, pathlib, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and importlib.util.find_spec('condacolab') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'])
if IN_COLAB:
    import condacolab
    condacolab.install()  # Colab restarts once; rerun this cell after restart.
    subprocess.check_call(['mamba', 'install', '-y', '-c', 'conda-forge', 'python=3.12', 'rdkit=2025.09.5', 'vina=1.2.7', 'meeko=0.7.1', 'pdbfixer=1.12', 'openmm=8.4.0', 'pydantic=2.11.7', 'httpx=0.28.1', 'pandas=2.2.3'])
else:
    print('Using the pre-provisioned pinned Linux chemistry environment.')
runtime_root = pathlib.Path('/content') if pathlib.Path('/content').is_dir() and os.access('/content', os.W_OK) else pathlib.Path.cwd() / '.terpedia-runtime'
runtime_root.mkdir(parents=True, exist_ok=True)
paper_root = pathlib.Path.cwd()
snapshot_dir = paper_root / 'notebooks' / 'terpedia-chemistry-snapshot'
if not snapshot_dir.exists():
    paper_root = runtime_root / 'terpedia-research-papers'
    if not paper_root.exists():
        subprocess.check_call(['git', 'clone', '--filter=blob:none', 'https://github.com/Terpedia/terpedia-research-papers.git', str(paper_root)])
    snapshot_dir = paper_root / 'notebooks' / 'terpedia-chemistry-snapshot'
snapshot_manifest = json.loads((snapshot_dir / 'manifest.json').read_text())
sys.path.insert(0, str(snapshot_dir))

In [ ]:
import asyncio, datetime as dt, hashlib, json, pathlib, platform, subprocess, os
from app.chemistry import parse_molecule
from app.docking import dock_hq_batch
from app.models import HqBatchDockingRequest
from app.study_data import STUDY_CONTROLS, STUDY_DATA_PROVENANCE, STUDY_LIGANDS, STUDY_TARGETS, terpene_names

BACKEND_COMMIT = snapshot_manifest['source_backend_commit']
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('colab-end-to-end-%Y%m%dT%H%M%SZ')
OUTPUT_DIR = runtime_root / 'terpedia-end-to-end-results'; OUTPUT_DIR.mkdir(exist_ok=True)
TARGETS = STUDY_TARGETS  # Set to a subset for a bounded pilot.
SEEDS = [20260717, 20260718, 20260719, 20260720, 20260721]
print({'run_id': RUN_ID, 'backend_commit': BACKEND_COMMIT, 'targets': [t['short_name'] for t in TARGETS], 'seeds': SEEDS, 'python': platform.python_version()})

## Run the validation-gated panel

The default configuration uses the full 23-terpene plus control panel for each target, five seeds, exhaustiveness 32, ten poses, and a two-angstrom redocking gate. Failures are retained in the receipt. This can require substantial CPU time in a free Colab session.

In [ ]:
def target_items(target):
    names = terpene_names() + STUDY_CONTROLS[target['short_name']]
    return [(parse_molecule(STUDY_LIGANDS[name]['smiles'], 'smiles'), name, {
        'pubchem_cid': STUDY_LIGANDS[name]['cid'],
        'snapshot': STUDY_DATA_PROVENANCE['snapshot_date'],
        'smiles_sha256': hashlib.sha256(STUDY_LIGANDS[name]['smiles'].encode()).hexdigest(),
    }) for name in names]

async def run_target(target):
    request = HqBatchDockingRequest(
        ligands=[{'structure': STUDY_LIGANDS[name]['smiles'], 'format': 'smiles', 'label': name} for name in terpene_names() + STUDY_CONTROLS[target['short_name']]],
        receptor={'pdb_id': target['pdb_id']}, engine='vina', scoring_functions=['vina', 'vinardo'],
        seeds=SEEDS, exhaustiveness=32, validation_exhaustiveness=32, poses=10, redocking_rmsd_threshold=2.0,
        require_redocking_pass=True, include_pose_artifacts=False
    )
    try:
        result = await dock_hq_batch(target_items(target), request)
        return {'status': result.get('status', 'unknown'), 'target': target, 'method': result.get('method'), 'validation': result.get('validation'), 'site': result.get('site'), 'receptor': result.get('receptor'), 'results': result.get('results', []), 'runs': result.get('runs', []), 'provenance': result.get('provenance')}
    except Exception as exc:
        return {'status': 'failed', 'target': target, 'error_type': type(exc).__name__, 'error': str(exc)}

results = []
for target in TARGETS:
    row = await run_target(target)
    results.append(row)
    (OUTPUT_DIR / f'{RUN_ID}-{target["short_name"]}.json').write_text(json.dumps(row, indent=2), encoding='utf-8')
for row in results:
    print(row['target']['short_name'], row['status'], row.get('validation', {}).get('median_rmsd_angstrom'), row.get('error', ''))

In [ ]:
# Seed-level uncertainty is descriptive, not an affinity confidence interval.
import statistics
uncertainty_summary = {}
for target_result in results:
    runs = target_result.get('runs', [])
    by_ligand = {}
    for run in runs:
        by_ligand.setdefault(run['ligand'], []).append(run['score_kcal_mol'])
    uncertainty_summary[target_result['target']['short_name']] = {label: {
        'n_seeds': len(values),
        'median_score_kcal_mol': statistics.median(values),
        'min_score_kcal_mol': min(values), 'max_score_kcal_mol': max(values),
        'median_absolute_deviation_kcal_mol': statistics.median([abs(value - statistics.median(values)) for value in values])
    } for label, values in by_ligand.items() if values}
print('Targets with seed-level uncertainty:', list(uncertainty_summary))

In [ ]:
receipt = {
    'schema': 'terpedia-end-to-end-docking-rerun/v1',
    'run_id': RUN_ID, 'backend_commit': BACKEND_COMMIT,
    'execution_status': 'completed' if all(row.get('status') == 'completed' for row in results) else 'completed_with_failures',
    'runtime': 'google_colab', 'computed_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    'protocol': {'engine': 'AutoDock Vina', 'scoring_functions': ['vina', 'vinardo'], 'seeds': SEEDS, 'exhaustiveness': 32, 'poses': 10, 'redocking_rmsd_threshold_angstrom': 2.0, 'no_affinity_inference': True},
    'input_provenance': STUDY_DATA_PROVENANCE, 'snapshot_manifest': snapshot_manifest, 'targets': results, 'uncertainty_summary': uncertainty_summary,
}
receipt_path = OUTPUT_DIR / f'{RUN_ID}.json'
receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print(receipt_path, receipt_path.stat().st_size, hashlib.sha256(receipt_path.read_bytes()).hexdigest())

## Required interpretation

Only targets that pass the redocking gate are eligible for candidate docking interpretation. A failed preparation or validation target has no valid candidate score. Compare ranks and validation outcomes, never Vina/MOE score magnitudes, and do not infer affinity, binding, safety, or clinical efficacy. Export the receipt alongside the report so every number remains linked to this Colab run.